### Resources
 
 'else' in a list comprehension<br>
 https://stackoverflow.com/questions/2951701/is-it-possible-to-use-else-in-a-list-comprehension
 
convert string to variable<br>
https://www.codeproject.com/Questions/1222606/Python-convert-string-and-variable-to-list-name

Apply function to each element of a list<br>
https://stackoverflow.com/questions/25082410/apply-function-to-each-element-of-a-list

In [1]:
# libraries, libraries!
import pandas as pd
import numpy as np # for 
import re # for RegEx
from re import search
from datetime import datetime

In [2]:
# set paths
pthPy         = r'P:\Working Folders\Hilton\Py' # path to Py stuff in working folder
pthCompliance = r'P:\Investment Operations\GRC\Compliance' # path to PIM Compliance folder
pthW          = r'P:\Working Folders\Hilton\W'

In [3]:
# get regex structure dataframes
structures = pthPy + r'\structures.xlsx'

# import the regex date formats
dfrgx = pd.read_excel(structures, sheet_name   = 'dates')

# import the list of accrual line items, #https://datatofish.com/pandas-dataframe-to-series/
#accr = pd.read_excel(structures, sheet_name = 'accr', usecols = ['accruals']).squeeze()
accr = pd.read_excel(structures, sheet_name    = 'accr', usecols = ['accruals'])
#accr_list = [x for e in list_of_accruals for x in e] # list of strings in accr

# import the regex CLN formats
clnrgx = pd.read_excel(structures, sheet_name  = 'cln')

# import the regex FRN formats
frnrgx = pd.read_excel(structures, sheet_name  = 'frn')

# import South African holidays
sa_hols = pd.read_excel(structures, sheet_name = 'hols', usecols = ['sa_hols'])

# import issuer regex formats
issrgx = pd.read_excel(structures, sheet_name  = 'issuers')

# create a list of the elements, excluding the NaNs, from the dataframe
rgxid = issrgx['id'].dropna() # https://stackoverflow.com/questions/46218652/python-pandas-unique-value-ignoring-nan

# import CLNs
clns = pd.read_excel(pthCompliance + r'\MCaps\CLNs.xlsx', sheet_name = 'CLN')

# import custodian accounts
sttlmnt = pd.read_excel(pthCompliance + r'\Daily\2A - Fund Codes, Breach Register.xlsx', sheet_name = 'Sttlmnt')

# import CMS circular
med = pd.read_excel(pthCompliance + r'\Medical Schemes\20220117 MSA Circ6 Categorisations.xlsx', sheet_name = 'JSEEquity31Dec2021')

# import the Reg 28 file with all issuer codes assigned a named issuer
Reg28_basis = pd.read_excel(pthW + r'\!Reg28Worx 30Jun2022.xlsx')

In [4]:
# utility function to open an excel file, .xls or .xlsx
def open_xl_file(file_name_and_path):
    import win32com.client as win32 # library to convert xls to xlsx
    excel = win32.gencache.EnsureDispatch('Excel.Application')
    excel.DisplayAlerts = False # suppress the warning dialogue
    excel.Workbooks.Open(file_name_and_path)
    excel.DisplayAlerts = True # unsuppress the warning dialogue

In [5]:
# function that extracts date from string
def datex(txt):
    try:
        for pattern in dfrgx['date_regex']:
            if re.search(pattern, txt.title()):
                break
        return datetime.strptime(re.search(pattern, txt.title()).group(),dfrgx.loc[dfrgx['date_regex'] == pattern].iat[0,1]).strftime("%d%b%Y")

    # manage exceptions:
    except ValueError as ve:
        print(f'ValueError {ve}')
    except TypeError as te:
        print(f'TypeError {te}')
    except AttributeError as ae:
        print(f'AttributeError {ae}')
        
#test the datex function
txt = "BNP Paribas Personal Finance SA Ltd FRN BPPF31 Jb3+95 17-Aug-13"
datex(txt)

'17Aug2013'

In [1]:
# functions to extract CLN, FRN, NCD, ILB

def cln(txt):
    for pattern in clnrgx['regex']:
        if re.search(pattern, txt.upper()): 
            #return clnrgx.loc[clnrgx['regex'] == pattern].iat[0,2] # note 'break' below the first 'for'   
            return 1
        break
        
def frn(txt):
    for pattern in frnrgx['regex']:
        if re.search(pattern, txt.upper()): 
            #return frnrgx.loc[frnrgx['regex'] == pattern].iat[0,2] # note 'break' below the first 'for' 
            return 1
        break
        
# import the regex ILB formats
ilbrgx = pd.read_excel(structures, sheet_name = 'ilb')
def ilb(txt):
    for pattern in ilbrgx['regex']:
        if re.search(pattern, txt.upper()): 
            #return ilbrgx.loc[ilbrgx['regex'] == pattern].iat[0,2] # note the 'break' below the first 'for'  
            return 1
        break       
        
txt = 'de Barge inflation asn'
print(frn(txt), type(frn(txt)), cln(txt), type(cln(txt)), ilb(txt), type(ilb(txt)))

NameError: name 'pd' is not defined

In [7]:
# function to derive ISSUER from INSTRUMENT DESCRIPTION ("i Issue Name" column)
def issuer_desc(txt):
    for pattern in issrgx['description']:
        if re.search(pattern, txt.upper()): 
            return issrgx.loc[issrgx['description'] == pattern].iat[0,2] # note 'break' within the for loop   
            break
        
test_txt_1 = 'The quick brown fox jumped mercedes over the lazy dog.'
print(issuer_desc(test_txt_1))

Mercedes-Benz South Africa (Pty) Ltd


In [8]:
# function to derive ISSUER from INSTRUMENT ID ("Primary Asset ID" column)
def issuer_id(txt):
    for pattern in rgxid:
        if re.search(pattern, txt.upper()): # check if the pattern exists in the txt
            return issrgx.loc[issrgx['id'] == pattern].iat[0,2] # note the 'break' within the for loop 
            break

test_txt_2 = 'SBT054 the lazy dog'
print(issuer_id(test_txt_2))

Standard Bank Group Ltd


In [9]:
# function to derive ACCRUAL from INSTRUMENT ID ("Primary Asset ID" column)
def accrual_id(txt):
    if accr['accruals'].str.contains(txt).any():
            return txt

test_txt_3 = 'INTWHT'
print(accrual_id(test_txt_3))

INTWHT


In [10]:
type(test_txt_3)

str

In [11]:
# read in a Reg 28 file from Eagle as a dataframe called 'example' to use as an input
example = pd.read_excel(pthW + r'\Reg 28 Report - PD.xlsx') # example.iloc[:,1:5]
fl = pthW + '\Reg 28 Report - PD.xlsx'

In [12]:
# function that combines issuer_desc() and issuer_id() to pick up ISSUER
def issuer(txt):
    if issuer_desc(txt) is None:
        return issuer_id(txt)
    else:
        return issuer_desc(txt)

txt1 = "The quick brown Southchester fox jumped over the lazy dog."
txt2 = "MTN023"
issuer(txt2)

'MTN Group Ltd'

In [13]:
# identify instrument attributes
# Map an if statement in Python: https://stackoverflow.com/questions/29247718/map-an-if-statement-in-python
# example['Date'] = example['i Issue Name'].map(lambda x: )
example['Issuer_Name']    = example['i Issue Name'].map(issuer_desc)
example['Issuer_ID']      = example['Primary Asset ID'].map(issuer_id)
example['Issuer_Accrual'] = example['Primary Asset ID'].map(accrual_id)
example['CLN']            = example['i Issue Name'].map(cln)
example['FRN']            = example['i Issue Name'].map(frn)
example['ILB']            = example['i Issue Name'].map(ilb)
example['Date']           = example['i Issue Name'].map(datex)

AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError 'NoneType' object has no attribute 'group'
AttributeError

In [14]:
# write the results to excel
example.to_excel(fl, index = False, sheet_name = 'R, b')
open_xl_file(fl) # open the file

In [ ]:
print(len(row['Primary Asset ID']))

In [16]:
for k in row['Primary Asset ID']:
    print(k)

NameError: name 'row' is not defined

In [39]:
#  function that creates a new column based on the values of the column colC
def categorise(row):  
    if   accrual_id(row['Primary Asset ID']):
        return 'A'
    elif issuer_desc(row['i Issue Name']):
        return 'B'
    elif issuer_id(row['Primary Asset ID']):
        return 'C'
    return 'D'

In [17]:
def func(row):
    return print(row['Primary Asset ID'], type(row['Primary Asset ID']), row['i Issue Name'], type(row['i Issue Name'], ))

In [41]:
example.apply(lambda row: func(row), axis=1)

AUDITFEE <class 'str'> Audit Fee <class 'str'>
INCDIST <class 'str'> Income Distribution <class 'str'>
INTRZAR <class 'str'> Interest on Cash ZAR <class 'str'>
INTWHT <class 'str'> Interest Withholding Tax <class 'str'>
MGMTFEE <class 'str'> Management Fee <class 'str'>
CASHNAD <class 'str'> Namibia Dollar <class 'str'>
OTHFEE <class 'str'> Other Fee <class 'str'>
PERFFEE <class 'str'> Performance Fee <class 'str'>
CASHZAR <class 'str'> South African Rand <class 'str'>
CASHUSD <class 'str'> US Dollar <class 'str'>
VARMARG <class 'str'> Variation Margin <class 'str'>
ABSATRS040822D <class 'str'> ABSA TRS 040822 D <class 'str'>
ABSATRS040822D <class 'str'> ABSA TRS 040822 D_P <class 'str'>
ABSATRS040822D <class 'str'> ABSA TRS 040822 D_R <class 'str'>
ABSATRS040822F <class 'str'> ABSA TRS 040822 F <class 'str'>
ABSATRS040822F <class 'str'> ABSA TRS 040822 F_P <class 'str'>
ABSATRS040822F <class 'str'> ABSA TRS 040822 F_R <class 'str'>
ABSATRS060223A <class 'str'> ABSA TRS SWAP 060223 A <

0      None
1      None
2      None
3      None
4      None
       ... 
362    None
363    None
364    None
365    None
366    None
Length: 367, dtype: object

In [16]:
df['Issuer'] = example.apply(lambda row: categorise(row), axis=1)
example

AttributeError: 'list' object has no attribute 'upper'

In [147]:
#  function that creates a new column based on the values of the column colC
def categorise(row):  
    if   row['colC'] > 0   and row['colC'] <= 99 :
        return 'A'
    elif row['colC'] > 100 and row['colC'] <= 199:
        return 'B'
    elif row['colC'] > 200 and row['colC'] <= 299:
        return 'C'
    return 'D'

# all you need to do is to pass the above method to apply() as a lambda expression
df['colF'] = df.apply(lambda row: categorise(row), axis=1)
df

In [61]:
# function that searches INSTRUMENT ID (Primary Asset ID) text for ISSUER
def issuer_id(txt):
    for pattern in rgxid:
        if re.search(pattern, txt.upper()): # check if the pattern exists in the txt
            return issrgx.loc[issrgx['id'] == pattern].iat[0,2] # note 'break' below 'return'   
            break

txt = 'SBT054 the lazy dog'
print(issuer_id(txt))

Standard Bank Group Ltd


In [64]:
# Remove square brackets from a List in Python
# https://bobbyhadz.com/blog/python-remove-square-brackets-from-list
list_of_accruals = accr.values.tolist()
accr_list = [x for e in list_of_accruals for x in e]
print(accr_list, type(accr_list))

['INCDIST', 'INTRZAR', 'INTWHT', 'VARMARG', 'AUDITFEE', 'MGMTFEE', 'PERFFEE', 'OTHFEE', 'CASHZAR', 'CASHCHF', 'CASHAUD', 'CASHEUR', 'CASHGBP', 'CASHNAD', 'CASHJPY', 'CASHUSD'] <class 'list'>


In [15]:
#  function that creates a new column based on the values of the column colC
def categorise(row):  
    if   row['Primary Asset ID'].map(accrual_id):
        return 'A'
    elif row['i Issue Name'].map(issuer_desc):
        return 'B'
    elif row['Primary Asset ID'].map(issuer_id):
        return 'C'
    return 'D'

In [139]:
# TEST 1
# function that combines issuer_accr(), issuer_id(), and issuer_desc() to pick up ISSUER

def issuer(txt):
    if accr['accruals'].str.contains(txt).any():    # Check if string is in a pandas dataframe
    # https://stackoverflow.com/questions/30944577/check-if-string-is-in-a-pandas-dataframe
        return txt
    elif issuer_desc(txt) is None:
        return issuer_id(txt)
    else:
        return issuer_desc(txt)

txt1 = "The quick brown Southchester fox jumped over the lazy dog."
txt2 = "MTN023"
txt3 = "INTRZAR"
print(issuer(txt1) + ', ' + issuer(txt2) + ', ' + issuer(txt3))

Southchester (RF) Ltd, MTN Group Ltd, INTRZAR


In [141]:
# TEST 2
# function that combines issuer_accr(), issuer_id(), and issuer_desc() to pick up ISSUER

def issuer2(txt):
    if issuer_desc(txt) is None:
        if accr['accruals'].str.contains(txt).any():
            return txt
        elif issuer_desc(txt) is None:
            return issuer_id(txt)
    else:
        return issuer_desc(txt)

txt1 = "The quick brown Southchester fox jumped over the lazy dog."
txt2 = "MTN023"
txt3 = "INTRZAR"
print(issuer2(txt1) + ', ' + issuer(txt2) + ', ' + issuer(txt3))

Southchester (RF) Ltd, MTN Group Ltd, INTRZAR
